# PyTorch Basics

In the previous notebook we built a neural network from scratch. We manually computed gradients, updated weights, and handled every detail. It was educational — but painful.

**PyTorch automates the painful parts:**
- **Autograd**: automatic differentiation — never compute gradients by hand again
- **nn.Module**: reusable building blocks for architectures
- **GPU support**: move tensors to GPU with one line
- **Dynamic computation graphs**: built on-the-fly, easy to debug with print statements

The core idea: you define the forward pass, PyTorch figures out the backward pass.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available() if hasattr(torch.backends, 'mps') else 'N/A'}")

---
## 1. Tensors — NumPy Arrays with Superpowers

Tensors are PyTorch's fundamental data structure. Think of them as numpy arrays that:
- Can live on GPU for parallel computation
- Track operations for automatic gradient computation

In [ ]:
t1 = torch.tensor([1, 2, 3])
t2 = torch.zeros(3, 4)
t3 = torch.randn(2, 3)
t4 = torch.arange(0, 10, 2)
t5 = torch.linspace(0, 1, 5)

print(f"From list:    {t1}  shape={t1.shape}  dtype={t1.dtype}")
print(f"Zeros:        shape={t2.shape}")
print(f"Random:       shape={t3.shape}")
print(f"Arange:       {t4}")
print(f"Linspace:     {t5}")

In [ ]:
t = torch.randn(2, 3, 4)
print(f"Original shape: {t.shape}")
print(f"Reshape:        {t.reshape(6, 4).shape}")
print(f"View:           {t.view(2, 12).shape}")
print(f"Transpose:      {t.permute(2, 0, 1).shape}")
print(f"Squeeze:        {torch.randn(1, 3, 1).squeeze().shape}")
print(f"Unsqueeze:      {torch.randn(3).unsqueeze(0).shape}")

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print(f"Add:      {a + b}")
print(f"Multiply: {a * b}")
print(f"Dot:      {a @ b}")
print(f"MatMul:   {torch.randn(2, 3) @ torch.randn(3, 4)} — shape (2,4)")
print(f"Sum:      {a.sum()}")
print(f"Mean:     {a.mean()}")

In [ ]:
np_array = np.array([1.0, 2.0, 3.0])
tensor_from_np = torch.from_numpy(np_array)
back_to_np = tensor_from_np.numpy()

print(f"NumPy → Tensor: {tensor_from_np}")
print(f"Tensor → NumPy: {back_to_np}")
print(f"\nThey share memory — modifying one changes the other:")
np_array[0] = 999
print(f"Changed np_array[0]=999, tensor now: {tensor_from_np}")

---
## 2. Autograd — Automatic Differentiation

This is PyTorch's killer feature. Set `requires_grad=True` on a tensor, do operations, call `.backward()`, and PyTorch computes all gradients for you.

No more hand-deriving chain rule. No more gradient bugs.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2 + 2 * x + 1

y.backward()

print(f"y = x² + 2x + 1")
print(f"x = {x.item()}")
print(f"y = {y.item()}")
print(f"dy/dx = 2x + 2 = {x.grad.item()}")
print(f"Expected: 2*3 + 2 = {2*3 + 2}  ✓")

In [ ]:
w = torch.tensor(0.5, requires_grad=True)
b = torch.tensor(0.1, requires_grad=True)
x_val = torch.tensor(2.0)
y_true = torch.tensor(1.0)

z = w * x_val + b
a = torch.sigmoid(z)
loss = -(y_true * torch.log(a) + (1 - y_true) * torch.log(1 - a))

loss.backward()

print(f"loss = {loss.item():.6f}")
print(f"∂loss/∂w = {w.grad.item():.6f}")
print(f"∂loss/∂b = {b.grad.item():.6f}")
print(f"\nThese match what we computed by hand in the previous notebook!")

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
funcs = {
    'x²': lambda x: x ** 2,
    'sin(x)': lambda x: torch.sin(x),
    'eˣ': lambda x: torch.exp(x),
    'log(x)': lambda x: torch.log(x),
}

print(f"Autograd handles any differentiable function (x={x.item()}):")
for name, fn in funcs.items():
    x_t = torch.tensor(2.0, requires_grad=True)
    y = fn(x_t)
    y.backward()
    print(f"  d({name})/dx = {x_t.grad.item():.4f}")

---
## 3. nn.Module — Building Neural Networks

PyTorch provides building blocks via `torch.nn`. The pattern:
1. Define layers in `__init__`
2. Define data flow in `forward`
3. PyTorch handles `backward` automatically

In [ ]:
linear = nn.Linear(3, 2)
print(f"Linear layer: 3 inputs → 2 outputs")
print(f"Weight shape: {linear.weight.shape}")
print(f"Bias shape:   {linear.bias.shape}")

x = torch.randn(5, 3)
out = linear(x)
print(f"\nInput shape:  {x.shape}")
print(f"Output shape: {out.shape}")

In [ ]:
model_sequential = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
    nn.Sigmoid()
)

print(model_sequential)
print(f"\nTotal parameters: {sum(p.numel() for p in model_sequential.parameters())}")

In [ ]:
class MoonsClassifier(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.net(x)

model = MoonsClassifier()
print(model)

x_test = torch.randn(3, 2)
print(f"\nForward pass: {model(x_test).detach().squeeze()}")

---
## 4. Loss Functions

PyTorch provides optimized loss functions that handle numerical stability automatically.

In [ ]:
y_pred_reg = torch.tensor([2.5, 3.2, 4.8])
y_true_reg = torch.tensor([3.0, 3.0, 5.0])
mse_loss = nn.MSELoss()
print(f"MSE Loss (regression): {mse_loss(y_pred_reg, y_true_reg).item():.4f}")

y_pred_bin = torch.tensor([0.9, 0.2, 0.8])
y_true_bin = torch.tensor([1.0, 0.0, 1.0])
bce_loss = nn.BCELoss()
print(f"BCE Loss (binary classification): {bce_loss(y_pred_bin, y_true_bin).item():.4f}")

y_pred_multi = torch.tensor([[2.0, 1.0, 0.1], [0.5, 2.5, 0.3]])
y_true_multi = torch.tensor([0, 1])
ce_loss = nn.CrossEntropyLoss()
print(f"CE Loss (multi-class): {ce_loss(y_pred_multi, y_true_multi).item():.4f}")

print("\n┌─────────────────┬──────────────────────────┬────────────────────────┐")
print("│ Loss Function   │ Use Case                 │ Output Activation      │")
print("├─────────────────┼──────────────────────────┼────────────────────────┤")
print("│ MSELoss         │ Regression               │ None (raw values)      │")
print("│ BCELoss         │ Binary classification    │ Sigmoid                │")
print("│ CrossEntropyLoss│ Multi-class              │ None (raw logits)      │")
print("└─────────────────┴──────────────────────────┴────────────────────────┘")

---
## 5. Optimizers — How Weights Get Updated

The optimizer takes gradients and decides how to update weights. The three-step dance every training step:

```python
optimizer.zero_grad()  # clear old gradients
loss.backward()        # compute new gradients
optimizer.step()       # update weights using gradients
```

In [ ]:
model_demo = nn.Linear(1, 1)

sgd = optim.SGD(model_demo.parameters(), lr=0.01)
adam = optim.Adam(model_demo.parameters(), lr=0.001)

print("SGD — vanilla gradient descent. Simple, but slow.")
print(f"  Parameters: lr={sgd.defaults['lr']}")
print(f"\nAdam — adaptive learning rate per parameter. Fast, works out of the box.")
print(f"  Parameters: lr={adam.defaults['lr']}, betas={adam.defaults['betas']}")
print(f"\nRule of thumb: start with Adam(lr=1e-3). Switch to SGD+momentum for fine-tuning.")

In [ ]:
torch.manual_seed(42)
X_simple = torch.linspace(-1, 1, 50).unsqueeze(1)
y_simple = 3 * X_simple + 2 + torch.randn_like(X_simple) * 0.3

model_lr = nn.Linear(1, 1)
optimizer = optim.SGD(model_lr.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

losses = []
for epoch in range(100):
    y_pred = model_lr(X_simple)
    loss = loss_fn(y_pred, y_simple)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())

w = model_lr.weight.item()
b = model_lr.bias.item()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(losses, color='#e74c3c', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss', fontweight='bold')
ax1.grid(True, alpha=0.3)

ax2.scatter(X_simple.numpy(), y_simple.numpy(), alpha=0.5, s=30, label='Data')
ax2.plot(X_simple.numpy(), model_lr(X_simple).detach().numpy(),
         color='#e74c3c', linewidth=2, label=f'Fit: y={w:.2f}x+{b:.2f}')
ax2.set_title('Linear Regression with PyTorch', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Learned: y = {w:.2f}x + {b:.2f}  (true: y = 3x + 2)")

---
## 6. DataLoader — Efficient Batching

For real datasets, we don't feed all data at once. We use **mini-batches**:
- Fits in memory (can't load 1M images at once)
- Noisy gradients act as regularization → better generalization
- GPU parallelism works best with batches

In [ ]:
X_data = torch.randn(100, 3)
y_data = torch.randint(0, 2, (100,)).float()

dataset = TensorDataset(X_data, y_data)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

print(f"Dataset size: {len(dataset)}")
print(f"Batch size: 16")
print(f"Batches per epoch: {len(dataloader)}")

for batch_idx, (X_batch, y_batch) in enumerate(dataloader):
    if batch_idx < 3:
        print(f"  Batch {batch_idx}: X shape={X_batch.shape}, y shape={y_batch.shape}")

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, n_samples=500):
        self.X = torch.randn(n_samples, 4)
        self.y = (self.X[:, 0] + self.X[:, 1] > 0).float()
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

custom_ds = CustomDataset(500)
custom_loader = DataLoader(custom_ds, batch_size=32, shuffle=True)

print(f"Custom dataset: {len(custom_ds)} samples")
X_sample, y_sample = custom_ds[0]
print(f"Single sample: X={X_sample}, y={y_sample}")

---
## 7. Full Example — Classify Moons Dataset

Let's put everything together: model, loss, optimizer, DataLoader, training loop, evaluation.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

X_np, y_np = make_moons(n_samples=500, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_np, y_np, test_size=0.2, random_state=42)

X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train).unsqueeze(1)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.FloatTensor(y_test).unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)

In [ ]:
class MoonsNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.net(x)

torch.manual_seed(42)
model = MoonsNet()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

train_losses = []
test_losses = []

for epoch in range(100):
    model.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    train_losses.append(epoch_loss / len(train_loader))
    
    model.eval()
    with torch.no_grad():
        test_pred = model(X_test_t)
        test_loss = criterion(test_pred, y_test_t)
        test_losses.append(test_loss.item())
    
    if epoch % 20 == 0:
        acc = ((test_pred > 0.5).float() == y_test_t).float().mean()
        print(f"  Epoch {epoch:3d} | Train Loss: {train_losses[-1]:.4f} | Test Loss: {test_losses[-1]:.4f} | Acc: {acc:.3f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses, label='Train', color='#e74c3c', linewidth=2)
ax1.plot(test_losses, label='Test', color='#3498db', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Test Loss', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

x_min, x_max = X_np[:, 0].min() - 0.5, X_np[:, 0].max() + 0.5
y_min, y_max = X_np[:, 1].min() - 0.5, X_np[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                     np.linspace(y_min, y_max, 200))
grid = torch.FloatTensor(np.c_[xx.ravel(), yy.ravel()])

model.eval()
with torch.no_grad():
    Z = model(grid).numpy().reshape(xx.shape)

ax2.contourf(xx, yy, Z, levels=50, cmap='RdYlGn', alpha=0.8)
ax2.scatter(X_test[:, 0], X_test[:, 1],
            c=['#e74c3c' if y == 0 else '#2ecc71' for y in y_test],
            edgecolors='black', s=40, label='Test data')
ax2.set_title('Decision Boundary', fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.show()

with torch.no_grad():
    final_acc = ((model(X_test_t) > 0.5).float() == y_test_t).float().mean()
print(f"Final test accuracy: {final_acc:.1%}")

---
## 8. GPU Acceleration

Moving computations to GPU can give 10-100x speedup for large models. The pattern is simple: move model and data to the same device.

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f"Using device: {device}")

model_gpu = MoonsNet().to(device)
x_gpu = torch.randn(32, 2).to(device)

with torch.no_grad():
    output = model_gpu(x_gpu)
print(f"Model on: {next(model_gpu.parameters()).device}")
print(f"Input on: {x_gpu.device}")
print(f"Output on: {output.device}")

output_cpu = output.cpu().numpy()
print(f"Back on CPU: {output_cpu.shape}")

In [ ]:
import time

sizes = [100, 1000, 5000]

for n in sizes:
    a_cpu = torch.randn(n, n)
    b_cpu = torch.randn(n, n)
    
    start = time.time()
    _ = a_cpu @ b_cpu
    cpu_time = time.time() - start
    
    if device.type != 'cpu':
        a_dev = a_cpu.to(device)
        b_dev = b_cpu.to(device)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        start = time.time()
        _ = a_dev @ b_dev
        if device.type == 'cuda':
            torch.cuda.synchronize()
        dev_time = time.time() - start
        print(f"  {n}x{n} matmul — CPU: {cpu_time:.4f}s, {device}: {dev_time:.4f}s, speedup: {cpu_time/dev_time:.1f}x")
    else:
        print(f"  {n}x{n} matmul — CPU: {cpu_time:.4f}s (no GPU available)")

---
## 9. Saving and Loading Models

Two approaches:
1. **Save state_dict** (recommended) — saves only weights, not architecture
2. **Save entire model** — saves everything, but brittle

In [ ]:
import tempfile, os

with tempfile.TemporaryDirectory() as tmpdir:
    path = os.path.join(tmpdir, 'model.pth')
    
    torch.save(model.state_dict(), path)
    print(f"Saved model state_dict ({os.path.getsize(path)} bytes)")
    
    loaded_model = MoonsNet()
    loaded_model.load_state_dict(torch.load(path, weights_only=True))
    loaded_model.eval()
    
    with torch.no_grad():
        orig_pred = model(X_test_t[:5])
        loaded_pred = loaded_model(X_test_t[:5])
    
    print(f"\nOriginal predictions:  {orig_pred.squeeze().numpy().round(3)}")
    print(f"Loaded predictions:    {loaded_pred.squeeze().numpy().round(3)}")
    print(f"Match: {torch.allclose(orig_pred, loaded_pred)} ✓")

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    path = os.path.join(tmpdir, 'checkpoint.pth')
    
    checkpoint = {
        'epoch': 100,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_losses': train_losses,
        'test_losses': test_losses,
    }
    torch.save(checkpoint, path)
    
    loaded = torch.load(path, weights_only=False)
    print(f"Resumed from epoch {loaded['epoch']}")
    print(f"Last train loss: {loaded['train_losses'][-1]:.4f}")
    print(f"\nCheckpoints let you resume training after interruption.")

---
## PyTorch Training Template

Every PyTorch project follows this pattern:

```python
# 1. Data
dataset = MyDataset(...)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

# 2. Model
model = MyModel().to(device)

# 3. Loss + Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# 4. Training loop
for epoch in range(n_epochs):
    model.train()
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        output = model(X_batch)        # forward
        loss = criterion(output, y_batch) # loss
        
        optimizer.zero_grad()           # clear grads
        loss.backward()                 # backward
        optimizer.step()                # update

# 5. Evaluate
model.eval()
with torch.no_grad():
    predictions = model(X_test)
```

You'll write this loop hundreds of times. Memorize it.